# Load dependencies

In [4]:
#%pip install notion_client
#%pip install openai

In [1]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1")

In [2]:
import os
from openai import OpenAI
from notion_client import Client
from dotenv import load_dotenv
import ast
import re


# From .env get notion_token
load_dotenv()
notion_token = os.getenv('NOTION_TOKEN')
notion = Client(auth=notion_token)

# Create a dict object to map categorydisplay to different language
categorydisplay_dict = {
    "일상기록": {"en": "Daily Life", "es": "Vida Diaria"},
    "의료정보학": {"en": "Clinical Informatics", "es": "Informática Clínica"},
    "연구일기": {"en": "Research", "es": "Investigación"},
    "공부일기": {"en": "Learning", "es": "Aprendizaje"},
}

# Define the OpenAI API key
client = OpenAI(
  api_key=os.getenv('OPENAI_API_KEY'),
)

In [3]:
def extract_markdown(blocks):
    """
    Extract Markdown content from Notion blocks.

    Parameters:
    blocks (list): List of Notion blocks.

    Returns:
    str: Markdown content.
    """
    markdown_lines = []

    for block in blocks['results']:
        block_type = block['type']

        block_content = block[block_type]
        text =''

        if block_type != 'image' and block_type != 'divider':

            for rt in block_content['rich_text']:
    
                #Check if this text is linked
                if rt['text']['link'] != None:
                    tmp = f"[{rt['text']['content']}]({rt['text']['link']['url']})"
                else:
                    tmp = f"{rt['text']['content']}"

                #Check if annotations says if text is bold/italic/strikethrough/underline/code/colored
                if rt['annotations']['underline'] == True:
                    tmp = f"<ins>{tmp}</ins>"
                if rt['annotations']['bold'] == True:
                    tmp = f"**{tmp}**"
                if rt['annotations']['italic'] == True:
                    tmp = f"*{tmp}*"
                if rt['annotations']['strikethrough'] == True:
                    tmp = f"~~{tmp}~~"
                if rt['annotations']['code'] == True:
                    tmp = f"`{tmp}`"
                if rt['annotations']['color'] != 'default':
                    tmp = f"<span style='color:{rt['annotations']['color']}'>{tmp}</span>"

                text += tmp

            # Add two spaces at the end of each line to create line breaks
            text += '  '

        if block_type == 'paragraph':
            markdown_lines.append(text)
        elif block_type == 'heading_1':
            markdown_lines.append(f"# {text}")
        elif block_type == 'heading_2':
            markdown_lines.append(f"## {text}")
        elif block_type == 'heading_3':
            markdown_lines.append(f"### {text}")
        elif block_type == 'bulleted_list_item':
            markdown_lines.append(f"- {text}")
        elif block_type == 'numbered_list_item':
            markdown_lines.append(f"1. {text}")
        elif block_type == 'to_do':
            checked = block_content['checked']
            markdown_lines.append(f"- [{'x' if checked else ' '}] {text}")
        elif block_type == 'quote':
            text = text.replace('\n', '\n> ')  # Add > at the beginning of each line
            markdown_lines.append(f"> {text}")
        elif block_type == 'code':
            language = block_content['language']
            markdown_lines.append(f"```{language}\n{text}\n```")
        elif block_type == 'callout':
            icon = block_content['icon']['emoji']
            markdown_lines.append(f"> {icon} {text}")
        elif block_type == 'divider':
            markdown_lines.append("---")
        elif block_type == 'image':
            # Suppose we only use external url for images... for convenience

            if 'external' in block_content:
                url = block_content['external']['url']
            elif 'file' in block_content:
                url = block_content['file']['url']
            else:
                url = ""
            #url = block_content['external']['url']
            #caption = block_content['caption'][0]['plain_text']
            caption = ""
            markdown_lines.append(f"![{caption}]({url})  ")

    return markdown_lines

def extract_notion_page_id(notion_url):
    """
    Extract the Notion page ID from a Notion URL.
    
    Parameters:
    notion_url (str): The URL of the Notion page.
    
    Returns:
    str: The extracted Notion page ID.
    """
    # Define the regular expression pattern to match the Notion page ID - geting 32-character string
    pattern = re.compile(r'([a-f0-9]{32})')
    
    # Search for the pattern in the Notion URL
    match = pattern.search(notion_url)
    
    if match:
        # Extract the page ID
        page_id = match.group(1)
        
        # Insert hyphens in the pattern 8-4-4-4-12
        formatted_page_id = f"{page_id[:8]}-{page_id[8:12]}-{page_id[12:16]}-{page_id[16:20]}-{page_id[20:]}"
        
        return formatted_page_id
    else:
        raise ValueError("Invalid Notion URL or page ID not found.")
    

# Create a code to get properties from Notion Page to formulate Front Matter for Jekyll
def extract_frontmatter(page_id):
    """
    Get the properties of a Notion page.
    
    Parameters:
    page_id (str): The ID of the Notion page.
    
    Returns:
    dict: The properties of the Notion page.

    * of note, for my Jekyll page, I used the following properties
        - title: title of post
        - date: date post is created. Instead of using date in Notion, I used the date that I manually input in Notion
        - tags: tags of the post
        - categories: categories of the post
        - categorydisplay: wierd name, I know. But this is for display purposes in Jekyll
        - lang: language of the post; either kr, en, or es; default is kr
        - thumbnail: thumbnail image of the post
        - subtitle: subtitle of the post
    """
    # Get the page data
    page_data = notion.pages.retrieve(page_id)
    

    # Get the filename of the page
    filename = page_data['properties']['filename']['rich_text'][0]['plain_text']
    # Get the title of the page
    title = page_data['properties']['title']['rich_text'][0]['plain_text']
    # Get the date of the page
    date = page_data['properties']['date']['date']['start']
    # Get the tags of the page
    tags = [ i['name'] for i in page_data['properties']['tags']['multi_select']]
    # get the categories of the page
    categories = page_data['properties']['categories']['select']['name']
    # Get the categorydisplay of the page
    categorydisplay = page_data['properties']['categorydisplay']['select']['name']
    # Get the lang of the page
    lang = page_data['properties']['lang']['select']['name']
    # Get the thumbnail of the page
    thumbnail = page_data['properties']['thumbnail']['files'][0]['name']
    # Get the subtitle of the page
    subtitle = page_data['properties']['subtitle']['rich_text'][0]['plain_text']
    
    # Create a dictionary of the properties
    properties = {
        'filename': filename,
        'title': title,
        'date': date,
        'tags': ' '.join(tags),
        'categories': categories,
        'categorydisplay': categorydisplay,
        'lang': lang,
        'thumbnail': thumbnail,
        'subtitle': subtitle
    }
    
    return properties

# Write Jekyll post Markdown file
def get_jekyll_post_from_notion_page(page_id):
    """
    Write a Jekyll post Markdown file.
    
    Parameters:
    page_id (str): The ID of the Notion page.

    Outputs:
    page_fm (dict): The front matter of the page.
    page_md (list): The markdown content of the page.

    """
    # Get the title of the page
    page_fm = extract_frontmatter(page_id)


    # Get the markdown content of the page
    page_md = []

    response = notion.blocks.children.list(page_id)
    page_md.extend(extract_markdown(response))

    has_more = response['has_more']
    next_cursor = response['next_cursor']
    while has_more:
        if next_cursor:
            
            response = notion.blocks.children.list(page_id, start_cursor = next_cursor)
            page_md.extend(extract_markdown(response))
            has_more = response['has_more']
            next_cursor = response['next_cursor']
        else:
            break

    #page_md = extract_markdown(notion.blocks.children.list(page_id))

    return page_fm, page_md

def write_jekyll_post_from_fm_md(page_fm,page_md, language = "kr", multilang_title = "", multilang_subtitle = ""):
    """
    Write a Jekyll post Markdown file.
    
    Parameters:
    page_fm (dict): The front matter of the page.
    page_md (list): The markdown content of the page.
    language (str): The language of the page. Default is 'kr'. Options are 'kr', 'en', 'es'.
    multilang_title (str): The translated title of the page. Default is "". Used in case of language is 'en' or 'es'.
    multilang_subtitle (str): The translated subtitle of the page. Default is "". Used in case of language is 'en' or 'es'.

    Outputs:
    front_matter (str): The front matter of the Markdown file.
    filename (str): The filename of the Markdown file.

    """

    # Define the filename of the Markdown file
    #title_as_filename = "".join([x if x.isalnum() else "_" for x in page_fm['title']])
    filename = f"{page_fm['date']}-{page_fm['filename']}.md"

    # Define the front matter of the Markdown file
    # If language is Korean, then use default front matter
    if language == 'kr':
        front_matter = f"""---
layout: post
permalink: /{page_fm['categories']}/:title/
title: "{page_fm['title']}"
date: {page_fm['date']} 00:00:00 -0400
tags: {page_fm['tags']}
categories: {page_fm['categories']}
categorydisplay: {page_fm['categorydisplay']}
lang: {page_fm['lang']}
thumbnail: {page_fm['thumbnail']}
subtitle: {page_fm['subtitle']}
---\n"""
    # If language == 'en' or 'es', then use multilang_title and multilang_subtitle
    else:
        front_matter = f"""---
layout: post
permalink: /{language}/{page_fm['categories']}/:title/
title: "{multilang_title}"
date: {page_fm['date']} 00:00:00 -0400
tags: {page_fm['tags']}
categories: {page_fm['categories']}
categorydisplay: {categorydisplay_dict[page_fm['categorydisplay']][language]}
lang: {language}
thumbnail: {page_fm['thumbnail']}
subtitle: {multilang_subtitle}
---\n"""
    

    # Write the Markdown content to the file
    # Write in directory ./_posts/{lang}/{categories}/{filename}
    with open(os.path.join('_posts', language , page_fm['categories'],filename), 'w') as file:
        file.write(front_matter)
        for line in page_md:
            file.write(f"{line}\n")
    
    # Will return page_fm and filename for reference
    print(f"Jekyll post Markdown file written: {os.path.join('_posts', language , page_fm['categories'],filename)}")
    
    return front_matter, filename

# Will Create a Code to translate pmd to English using OpenAI

def translate_markdown(markdown_string, frontmatter_dict, target_language):
    """
    Translate markdown content to the target language using OpenAI.

    Parameters:
    pmd (list): List of markdown strings.
    target_language (str): Target language for translation.

    Returns:
    str: Translated markdown content.
    """
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a translator of a Jekyll blog. Your job is to translate contents in markdown into given language; keep all markdown syntax."},
            {"role": "user", "content": f"Translate the following Korean markdown page to {target_language}: {markdown_string}"}
        ]
    )

    title_completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Translate following Korean phrase to given language."},
            {"role": "user", "content": f"Translate this to {target_language}: {frontmatter_dict['title']}"}
        ]
    )

    subtitle_completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Translate following Korean phrase to given language."},
            {"role": "user", "content": f"Translate this to {target_language}: {frontmatter_dict['subtitle']}"}
        ]
    )

    return completion.choices[0].message, title_completion.choices[0].message, subtitle_completion.choices[0].message

In [4]:

def translate_markdown_llama(markdown_string, frontmatter_dict, target_language):
    """
    Translate markdown content to the target language using OpenAI.

    Parameters:
    pmd (list): List of markdown strings.
    target_language (str): Target language for translation.

    Returns:
    str: Translated markdown content.
    """

    md_prompt = f""" 
    You are a translator for a Jekyll blog. Your task is to translate the following Korean markdown content into {target_language}, preserving all markdown syntax.
    
    Instructions:
    - Translate the text content from Korean to {target_language}.
    - Preserve all markdown formatting and syntax exactly as in the original.
    - Do not translate or change any code inside code blocks denoted by (triple backticks).
    - Exception: Translate comments within code blocks (e.g., lines starting with #, //, or enclosed in /* */) into {target_language}.
    - Do not add, remove, or alter any markdown elements.
    - Do not include any explanations, comments, or additional text.
    - Output only the translated markdown content.
        
    Content to translate:
    {markdown_string}
    """

    md_response = llm.invoke(md_prompt, temperature=0.0)

    title_prompt = f""" 
    You are a translator for a Jekyll blog. Your task is to translate the following Korean phrase to {target_language}.
    
    Translate the following blog title to {target_language}: {frontmatter_dict['title']}
    Do not include any explanations, comments, or additional text. Just return the translated title.
    """

    title_response = llm.invoke(title_prompt, temperature=0.0)

    subtitle_prompt = f""" 
    You are a translator for a Jekyll blog. Your task is to translate the following Korean phrase to {target_language}.
    
    Translate the following post subtitle to {target_language}: {frontmatter_dict['subtitle']}
    Do not include any explanations, comments, or additional text. Just return the translated subtitle.
    """

    title_response = llm.invoke(subtitle_prompt, temperature=0.0)

    return md_response, title_response, title_response

# Execute

- Define notion page url
- Create root markdown file
- Use the file to create Spanish/English pages
- Store them accordingly

In [33]:
#test = notion.blocks.children.list(extract_notion_page_id("https://www.notion.so/seungwooklee/Notion-Jekyll-URL-86e9ef81af6e4cefae24c0c733ce6853"))

In [35]:
page_url = "https://seungwooklee.notion.site/feat-ChatGPT-Llama-3-1-108be74ccd5c8057a46fc32229141eae"

# Example usage
#notion_url = "https://www.notion.so/your-page-title-a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6"
page_id = extract_notion_page_id(page_url)
print("Notion Page ID:", page_id)

# Get the front matter and markdown content of the Notion page
pfm, pmd = get_jekyll_post_from_notion_page(page_id)
# Translate the markdown content to English
translated_content_en, en_title, en_subtitle = translate_markdown(' '.join(pmd), pfm, 'English')
en_md_list = translated_content_en.content.split('\n')
# Translate the markdown content to Spanish
translated_content_es, es_title, es_subtitle = translate_markdown(' '.join(pmd), pfm, 'Spanish')
es_md_list = translated_content_es.content.split('\n')


# Write the Jekyll post Markdown file in Korean
_,_ = write_jekyll_post_from_fm_md(pfm,pmd)
# Write the English translated Jekyll post Markdown file
_,_ = write_jekyll_post_from_fm_md(pfm,en_md_list, language='en', multilang_title=en_title.content, multilang_subtitle=en_subtitle.content)
# Write the Spanish translated Jekyll post Markdown file
_,_ = write_jekyll_post_from_fm_md(pfm,es_md_list, language='es', multilang_title=es_title.content, multilang_subtitle=es_subtitle.content)


Notion Page ID: 108be74c-cd5c-8057-a46f-c32229141eae


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

pmd

# Execute with Llama 3.1

In [33]:
def testl3(text):
    
    test = f""" 
    Translate the following Korean text into English while preserving all markdown syntax. Translate only the content in Korean, and do not change or remove any markdown elements:
    - Preserve all markdown formatting and syntax exactly as in the original.
    - Do not translate or change any code inside code blocks denoted by (triple backticks).
    - Exception: Translate comments within code blocks (e.g., lines starting with #, //, or enclosed in /* */) into English.
    - Do not add, remove, or alter any markdown elements.
    - Do not include any explanations, comments, or additional text.
    - Output only the translated markdown content.
    - Don't add any additional texts like 'Here is the translation of...'
    {text}
    """
    
    response = llm.invoke(test, temperature=0.0)
    
    return response

tt = testl3(pmd[0])

print(tt)

Here is the translation of the Korean text into English, preserving all markdown syntax:

- Previous posts discussed **[Making a multi-language supported Jekyll post](https://slee333.github.io/learning/jekyll-multilang-without-plugin/)**, and **[How to easily post to Jekyll blog using Notion](https://slee333.github.io/learning/Jeykll-notion-to-jekyll/)**.


In [27]:
pmd[0]

'이전 포스트 들에서는 **[다중 언어 지원 Jekyll post 만들기](https://slee333.github.io/learning/jekyll-multilang-without-plugin/)**, 그리고 **[Notion을 이용해서 Jekyll 블로그 포스팅을 쉽게 하는 법](https://slee333.github.io/learning/Jeykll-notion-to-jekyll/)**을 다루어 보았다.  '

In [26]:
r_t

"It sounds like you've successfully set up a translation function using the LLaMA 3.1 model and are now able to create multilingual posts on your blog with ease.\n\nI must say, I'm impressed by your creative use of the LLaMA API and the way you've tailored it to your specific needs. It's great that you're experimenting with different models and techniques to find what works best for you.\n\nAs you continue to refine your process and add new features to your blog, I encourage you to share more about your experiences and any challenges you may face. This can help others who are interested in similar projects or facing similar issues.\n\nAlso, if you don't mind me asking, how do you plan to handle any potential errors or discrepancies that might arise from using an AI-powered translation model? For example, what would happen if the LLaMA model misinterprets a phrase or sentence?\n\nFinally, I'm curious - have you considered exploring other language models or APIs that might offer better p

In [5]:
page_url = "https://seungwooklee.notion.site/feat-ChatGPT-Llama-3-1-108be74ccd5c8057a46fc32229141eae"

# Example usage
#notion_url = "https://www.notion.so/your-page-title-a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6"
page_id = extract_notion_page_id(page_url)
print("Notion Page ID:", page_id)

# Get the front matter and markdown content of the Notion page
pfm, pmd = get_jekyll_post_from_notion_page(page_id)
# Translate the markdown content to English
translated_content_en, en_title, en_subtitle = translate_markdown_llama(' '.join(pmd), pfm, 'English')
en_md_list = translated_content_en.content.split('\n')
# Translate the markdown content to Spanish
translated_content_es, es_title, es_subtitle = translate_markdown_llama(' '.join(pmd), pfm, 'Spanish')
es_md_list = translated_content_es.content.split('\n')


# Write the Jekyll post Markdown file in Korean
_,_ = write_jekyll_post_from_fm_md(pfm,pmd)
# Write the English translated Jekyll post Markdown file
_,_ = write_jekyll_post_from_fm_md(pfm,en_md_list, language='en', multilang_title=en_title.content, multilang_subtitle=en_subtitle.content)
# Write the Spanish translated Jekyll post Markdown file
_,_ = write_jekyll_post_from_fm_md(pfm,es_md_list, language='es', multilang_title=es_title.content, multilang_subtitle=es_subtitle.content)


Notion Page ID: 108be74c-cd5c-8057-a46f-c32229141eae


AttributeError: 'str' object has no attribute 'content'